# AML

## 1. Import packages

In [ ]:
import pandas as pd
import numpy as np
import os
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import glob
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
import seaborn as sns
import joblib

Save the dataset path

In [ ]:
data_path = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"

for root, dirs, files in os.walk(data_path):
    print(f"\nFolder: {root}")
    for file in files:
        print(f"  - {file}")

As we can see the IBM AML dataset isn't just one CSV. It gives us transactions, accounts, and pattern information.

For this project, I'll use the HI-Medium version as the main dataset:

## 2. Load the Dataset

In [ ]:
data_path = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"

trans_file = f"{data_path}/HI-Medium_Trans.csv"

# Read only the first 5 rows
sample = pd.read_csv(trans_file, nrows=5)

print("Columns:")
print(sample.columns.tolist())

print("\nFirst 5 rows:")
display(sample)

Notice these first transactions:

From Bank = 20
To Bank   = 20

and:

From Bank = 3196
To Bank   = 3196

That means not every transaction is cross-border.

The project description says:

Analyze international money transfers for unusual patterns.

So we should not assume every transaction is international.

Instead, we'll investigate the relationship between:

From Bank -> Account -> Transaction -> Account.1 -> To Bank
and determine what characteristics can indicate cross-border activity.

There are alsoextremely large transaction compared with the first few rows.

In [ ]:
trans_file = f"{data_path}/HI-Medium_Trans.csv"

file_size_gb = os.path.getsize(trans_file) / (1024**3)

print(f"File size: {file_size_gb:.2f} GB")

## 3. EDA

In [ ]:
chunk_size = 1_000_000

total_rows = 0
laundering_count = 0

for chunk in pd.read_csv(trans_file, chunksize=chunk_size):
    total_rows += len(chunk)
    laundering_count += chunk["Is Laundering"].sum()

print(f"Total transactions: {total_rows:,}")
print(f"Laundering transactions: {laundering_count:,}")
print(f"Laundering percentage: {(laundering_count / total_rows) * 100:.4f}%")

Now we know
| Metric                  |         Result |
| ----------------------- | -------------: |
| File size               |    **2.82 GB** |
| Transactions            | **31,898,238** |
| Laundering transactions |     **35,230** |
| Normal transactions     | **31,863,008** |
| Laundering rate         |    **0.1104%** |


That means only about 1 out of every 905 transactions is labeled laundering. This is a highly imbalanced classification problem. So later, accuracy cannot be the main evaluation metric.

From the collumns from earlier there is no:
- Sender Country
- Receiver Country

So, we need to understand how the IBM dataset represents international/cross-border activity.

In [ ]:
accounts_file = f"{data_path}/HI-Medium_accounts.csv"

accounts_sample = pd.read_csv(accounts_file, nrows=10)

print("Columns:")
print(accounts_sample.columns.tolist())

display(accounts_sample)

HI-Medium_accounts.csv contains bank/account information, and the bank names themselves reveal the country. So I can derive country information from the bank names and combine it with the transaction data.

In [ ]:
accounts_sample = pd.read_csv(accounts_file, nrows=1000)

print(accounts_sample["Bank Name"].head(20).to_string())

So country name cannot be simply extracted from Bank Name. Some banks clearly contain a country name while others don't.

In [ ]:
patterns_file = f"{data_path}/HI-Medium_Patterns.txt"

with open(patterns_file, "r", encoding="utf-8") as f:
    for i in range(100):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())

The HI-Medium_Patterns.txt file shows that the dataset was deliberately generated with several AML typologies—different ways money can be moved to create suspicious transaction structures.

| Pattern            | What it means conceptually                                                          |
| ------------------ | ----------------------------------------------------------------------------------- |
| **STACK**          | Money is transferred through a sequence of accounts                                 |
| **CYCLE**          | Money moves through several accounts and eventually returns to the starting account |
| **FAN-IN**         | Many accounts send money into one account                                           |
| **GATHER-SCATTER** | Many accounts feed into one account, which then distributes money to many others    |
| **BIPARTITE**      | Multiple senders and receivers form a connected transaction structure               |


In [ ]:
payment_formats = set()
payment_currencies = set()
receiving_currencies = set()

from_banks = set()
to_banks = set()

laundering_counts = {0: 0, 1: 0}

for chunk in pd.read_csv(trans_file, chunksize=1_000_000):

    payment_formats.update(chunk["Payment Format"].dropna().unique())
    payment_currencies.update(chunk["Payment Currency"].dropna().unique())
    receiving_currencies.update(chunk["Receiving Currency"].dropna().unique())

    from_banks.update(chunk["From Bank"].dropna().unique())
    to_banks.update(chunk["To Bank"].dropna().unique())

    counts = chunk["Is Laundering"].value_counts()

    for label, count in counts.items():
        laundering_counts[int(label)] += count

print("Payment Formats:")
print(sorted(payment_formats))

print("\nPayment Currencies:")
print(sorted(payment_currencies))

print("\nReceiving Currencies:")
print(sorted(receiving_currencies))

print("\nUnique From Banks:", len(from_banks))
print("Unique To Banks:", len(to_banks))

print("\nLaundering distribution:")
print(laundering_counts)

This tells us there are:
- 7 payment formats
- 15 currencies
- The transaction network is huge
- The class imbalance is confirmed


### Analyze categorical distributions

In [ ]:
payment_format_counts = Counter()
payment_format_laundering = Counter()

payment_currency_counts = Counter()
payment_currency_laundering = Counter()

receiving_currency_counts = Counter()
receiving_currency_laundering = Counter()

for chunk in pd.read_csv(trans_file, chunksize=1_000_000):

    # Payment format
    payment_format_counts.update(
        chunk["Payment Format"].dropna()
    )

    laundering = chunk[chunk["Is Laundering"] == 1]

    payment_format_laundering.update(
        laundering["Payment Format"].dropna()
    )

    # Payment currency
    payment_currency_counts.update(
        chunk["Payment Currency"].dropna()
    )

    payment_currency_laundering.update(
        laundering["Payment Currency"].dropna()
    )

    # Receiving currency
    receiving_currency_counts.update(
        chunk["Receiving Currency"].dropna()
    )

    receiving_currency_laundering.update(
        laundering["Receiving Currency"].dropna()
    )


print("=== PAYMENT FORMAT ===")
for fmt, count in payment_format_counts.most_common():
    laundering_count = payment_format_laundering[fmt]
    rate = (laundering_count / count) * 100
    print(f"{fmt:15} {count:12,} total | "
          f"{laundering_count:8,} laundering | "
          f"{rate:.4f}%")

print("\n=== PAYMENT CURRENCY ===")
for currency, count in payment_currency_counts.most_common():
    laundering_count = payment_currency_laundering[currency]
    rate = (laundering_count / count) * 100
    print(f"{currency:20} {count:12,} total | "
          f"{laundering_count:8,} laundering | "
          f"{rate:.4f}%")

print("\n=== RECEIVING CURRENCY ===")
for currency, count in receiving_currency_counts.most_common():
    laundering_count = receiving_currency_laundering[currency]
    rate = (laundering_count / count) * 100
    print(f"{currency:20} {count:12,} total | "
          f"{laundering_count:8,} laundering | "
          f"{rate:.4f}%")

Within the HI-Medium synthetic dataset, laundering-labeled transactions are disproportionately concentrated in ACH transactions.

Frequency and risk rate are different things.
The US Dollar has the most laundering transactions: 14,292
but that's partly because USD is extremely common.
The UK Pound has fewer laundering transactions: 1,629
but a higher laundering rate: 0.1599%

This distinction will become very important later. We should distinguish between Absolute suspicious volume and Suspicious rate relative to transaction volume.

Payment and receiving currency are almost identical. That suggests that, in the laundering examples we've seen, the payment and receiving currencies are often the same. But the overall transaction counts differ slightly.

### Understanding the time dimension

In [ ]:
min_time = None
max_time = None

daily_counts = {}
daily_laundering = {}

for chunk in pd.read_csv(
    trans_file,
    usecols=["Timestamp", "Is Laundering"],
    chunksize=1_000_000
):
    chunk["Timestamp"] = pd.to_datetime(chunk["Timestamp"])

    chunk["Date"] = chunk["Timestamp"].dt.date

    # Total transactions per day
    counts = chunk["Date"].value_counts()
    for date, count in counts.items():
        daily_counts[date] = daily_counts.get(date, 0) + count

    # Laundering transactions per day
    laundering = chunk[chunk["Is Laundering"] == 1]
    laundering_counts = laundering["Date"].value_counts()

    for date, count in laundering_counts.items():
        daily_laundering[date] = daily_laundering.get(date, 0) + count

    chunk_min = chunk["Timestamp"].min()
    chunk_max = chunk["Timestamp"].max()

    if min_time is None or chunk_min < min_time:
        min_time = chunk_min

    if max_time is None or chunk_max > max_time:
        max_time = chunk_max


print("Earliest transaction:", min_time)
print("Latest transaction:", max_time)

print("\nNumber of days:", len(daily_counts))

print("\nFirst 10 daily transaction counts:")
for date in sorted(daily_counts)[:10]:
    print(date, daily_counts[date])

print("\nFirst 10 daily laundering counts:")
for date in sorted(daily_laundering)[:10]:
    print(date, daily_laundering[date])

The dataset covers only 28 days.

In [ ]:
amount_stats = {
    "count": 0,
    "sum": 0.0,
    "sum_squared": 0.0,
    "min": np.inf,
    "max": -np.inf
}

# For approximate percentiles, collect a sample
amount_sample = []

laundering_amounts = []
normal_amounts = []

rng = np.random.default_rng(42)

for chunk in pd.read_csv(
    trans_file,
    usecols=["Amount Paid", "Is Laundering"],
    chunksize=1_000_000
):
    amounts = chunk["Amount Paid"].dropna()

    amount_stats["count"] += len(amounts)
    amount_stats["sum"] += amounts.sum()
    amount_stats["sum_squared"] += (amounts ** 2).sum()
    amount_stats["min"] = min(amount_stats["min"], amounts.min())
    amount_stats["max"] = max(amount_stats["max"], amounts.max())

    # Random sample for percentiles
    sample_size = min(10_000, len(amounts))
    amount_sample.extend(
        amounts.sample(
            n=sample_size,
            random_state=42
        ).tolist()
    )

    laundering_amounts.extend(
        chunk.loc[
            chunk["Is Laundering"] == 1,
            "Amount Paid"
        ].dropna().tolist()
    )

    normal_chunk = chunk.loc[
        chunk["Is Laundering"] == 0,
        "Amount Paid"
    ].dropna()

    # Smaller sample of normal transactions
    sample_size = min(5_000, len(normal_chunk))

    normal_amounts.extend(
        normal_chunk.sample(
            n=sample_size,
            random_state=42
        ).tolist()
    )

mean_amount = amount_stats["sum"] / amount_stats["count"]

print("=== ALL TRANSACTIONS ===")
print(f"Count: {amount_stats['count']:,}")
print(f"Minimum: {amount_stats['min']:,.2f}")
print(f"Maximum: {amount_stats['max']:,.2f}")
print(f"Mean: {mean_amount:,.2f}")

print("\nApproximate percentiles:")
for p in [25, 50, 75, 90, 95, 99, 99.9]:
    print(f"{p}%: {np.percentile(amount_sample, p):,.2f}")

print("\n=== LAUNDERING TRANSACTIONS ===")
print(f"Count: {len(laundering_amounts):,}")
print(f"Mean: {np.mean(laundering_amounts):,.2f}")
print(f"Median: {np.median(laundering_amounts):,.2f}")
print(f"Minimum: {np.min(laundering_amounts):,.2f}")
print(f"Maximum: {np.max(laundering_amounts):,.2f}")

print("\n=== NORMAL TRANSACTIONS SAMPLE ===")
print(f"Sample size: {len(normal_amounts):,}")
print(f"Mean: {np.mean(normal_amounts):,.2f}")
print(f"Median: {np.median(normal_amounts):,.2f}")
print(f"Minimum: {np.min(normal_amounts):,.2f}")
print(f"Maximum: {np.max(normal_amounts):,.2f}")

The amount distribution is extremely skewed and the laundering distribution also tells that laundering transactions are not simply very large transactions. So we need behavioral and contextual features.

Both normal and laundering transactions have 0 value which needs further analysis.

Note: the percentile calculation is approximate, because sampling is used rather than sorting all 31.9M values. That's intentional for memory efficiency.

Currency conversion / exchange relationship could be especially relevant to our cross-border transaction analysis.

Now lets analyze how many unique accounts exist, and how are transactions distributed across them?

In [ ]:
account_counts = set()
sender_counts = {}
receiver_counts = {}

laundering_accounts = set()

for chunk in pd.read_csv(
    trans_file,
    usecols=[
        "Account",
        "Account.1",
        "Is Laundering"
    ],
    chunksize=1_000_000
):

    # Unique accounts
    account_counts.update(chunk["Account"].dropna())
    account_counts.update(chunk["Account.1"].dropna())

    # Sender frequency
    sender_freq = chunk["Account"].value_counts()

    for account, count in sender_freq.items():
        sender_counts[account] = sender_counts.get(account, 0) + count

    # Receiver frequency
    receiver_freq = chunk["Account.1"].value_counts()

    for account, count in receiver_freq.items():
        receiver_counts[account] = receiver_counts.get(account, 0) + count

    # Accounts appearing in laundering transactions
    laundering = chunk[chunk["Is Laundering"] == 1]

    laundering_accounts.update(
        laundering["Account"].dropna()
    )

    laundering_accounts.update(
        laundering["Account.1"].dropna()
    )


print("Unique accounts:", len(account_counts))

print("Accounts appearing in laundering:",
      len(laundering_accounts))

print("\nTop 20 sender accounts:")
for account, count in sorted(
    sender_counts.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]:
    print(account, count)

print("\nTop 20 receiver accounts:")
for account, count in sorted(
    receiver_counts.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]:
    print(account, count)

This suggests there may be different roles within the synthetic transaction network.

41,857 accounts are connected to laundering transactions that's:

41,857 / 2,076,999 ≈ 2.01%

So approximately 2% of the unique accounts appear in at least one laundering-labeled transaction.

### Investigate the account dataset
Let's understand how many accounts, banks, entities, and entity types exist in that file.

In [ ]:
accounts_df = pd.read_csv(accounts_file)

print("Shape:", accounts_df.shape)

print("\nColumns:")
print(accounts_df.columns.tolist())

print("\nData types:")
print(accounts_df.dtypes)

print("\nUnique Bank IDs:",
      accounts_df["Bank ID"].nunique())

print("Unique Account Numbers:",
      accounts_df["Account Number"].nunique())

print("Unique Entity IDs:",
      accounts_df["Entity ID"].nunique())

print("Unique Entity Names:",
      accounts_df["Entity Name"].nunique())

print("\nEntity types:")
print(accounts_df["Entity Name"].value_counts().head(30))

We have 2,087,762 unique accounts and 668,138 unique entities

That means an entity can have multiple accounts.

We also have:

Rows:                   2,087,786
Unique Account Numbers: 2,087,762

Difference: 24 duplicate account numbers.

### Entity types and account distribution

In [ ]:
# Extract the entity type from Entity Name
accounts_df["Entity Type"] = (
    accounts_df["Entity Name"]
    .str.replace(r"\s+#\d+$", "", regex=True)
)

print("=== ENTITY TYPES ===")
print(accounts_df["Entity Type"].value_counts())

print("\n=== NUMBER OF UNIQUE ENTITIES BY TYPE ===")
print(
    accounts_df
    .groupby("Entity Type")["Entity ID"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n=== ACCOUNTS PER ENTITY ===")

accounts_per_entity = (
    accounts_df
    .groupby("Entity ID")["Account Number"]
    .nunique()
)

print(accounts_per_entity.describe())

print("\nEntities with the most accounts:")
print(
    accounts_per_entity
    .sort_values(ascending=False)
    .head(20)
)

From this we can see:
- There are 6 entity types
- The entity is structured
Earlier we saw some accounts have such extreme transaction counts. Now we discovered the dataset contains entities with thousands of accounts, including a few huge entities.
That means we need to distinguish:
- Account-level behavior
- Entity-level behavior


The account metadata may contain a synthetic representation of country-related accounts.

Now lets' see the actual relationship between accounts and entities

In [ ]:
print(
    accounts_df
    .groupby("Entity Type")["Account Number"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nAccounts per entity by entity type:")

entity_account_stats = (
    accounts_df
    .groupby(["Entity Type", "Entity ID"])["Account Number"]
    .nunique()
    .groupby(level=0)
    .describe()
)

print(entity_account_stats)

In [ ]:
print("\n=== COUNTRY ENTITIES ===")
print(
    accounts_df[
        accounts_df["Entity Type"] == "Country"
    ].head(20)
)

print("\n=== DIRECT ENTITIES ===")
print(
    accounts_df[
        accounts_df["Entity Type"] == "Direct"
    ].head(20)
)

This suggests that Bank Name can potentially be parsed to derive a bank's country. But, before creating From Country and To Country, we need to determine how reliably bank names encode country.

Entity type and bank country are different concepts. we should keep them separate.

Before moving forward lets see if bank county can be derived reliably.

In [ ]:
print("Unique bank names:", accounts_df["Bank Name"].nunique())

print("\nSample bank names:")
print(
    accounts_df["Bank Name"]
    .drop_duplicates()
    .sample(100, random_state=42)
    .to_list()
)

In [ ]:
print("\nBank names containing country-like names:")

countries = [
    "China", "Spain", "Mexico", "Switzerland", "Italy",
    "Russia", "Finland", "Israel", "Australia", "Brazil",
    "Germany", "France", "Saudi Arabia", "Canada", "UK",
    "United Kingdom", "Japan", "India", "Argentina",
    "Netherlands", "Turkey", "South Africa"
]

for country in countries:
    count = accounts_df["Bank Name"].str.contains(
        country,
        case=False,
        na=False
    ).sum()

    print(f"{country:20} {count:,}")

This confirms that the bank-name structure is useful for country identification, but it should be done carefully.

In [ ]:
print("Unique accounts:",
      accounts_df["Account Number"].nunique())

print("Duplicate account numbers:",
      accounts_df["Account Number"].duplicated().sum())

print("Unique banks:",
      accounts_df["Bank ID"].nunique())

print("Unique entities:",
      accounts_df["Entity ID"].nunique())

## 4. Data Cleaning
cleaning can be done in two stages
- Stage 1 — Transaction-level quality
- Stage 2 — Relational integrity

In [ ]:
missing_counts = {}
invalid_amounts = {
    "received_negative": 0,
    "received_zero": 0,
    "received_nan": 0,
    "paid_negative": 0,
    "paid_zero": 0,
    "paid_nan": 0
}

invalid_labels = {}
invalid_formats = set()
invalid_payment_currencies = set()
invalid_receiving_currencies = set()

total_rows = 0
duplicate_rows = 0

expected_formats = {
    "ACH",
    "Bitcoin",
    "Cash",
    "Cheque",
    "Credit Card",
    "Reinvestment",
    "Wire"
}

expected_currencies = {
    "Australian Dollar",
    "Bitcoin",
    "Brazil Real",
    "Canadian Dollar",
    "Euro",
    "Mexican Peso",
    "Ruble",
    "Rupee",
    "Saudi Riyal",
    "Shekel",
    "Swiss Franc",
    "UK Pound",
    "US Dollar",
    "Yen",
    "Yuan"
}

for chunk in pd.read_csv(
    trans_file,
    chunksize=1_000_000
):

    total_rows += len(chunk)

    # Missing values
    for col in chunk.columns:
        missing_counts[col] = (
            missing_counts.get(col, 0)
            + chunk[col].isna().sum()
        )

    # Amount checks
    received = pd.to_numeric(
        chunk["Amount Received"],
        errors="coerce"
    )

    paid = pd.to_numeric(
        chunk["Amount Paid"],
        errors="coerce"
    )

    invalid_amounts["received_negative"] += (received < 0).sum()
    invalid_amounts["received_zero"] += (received == 0).sum()
    invalid_amounts["received_nan"] += received.isna().sum()

    invalid_amounts["paid_negative"] += (paid < 0).sum()
    invalid_amounts["paid_zero"] += (paid == 0).sum()
    invalid_amounts["paid_nan"] += paid.isna().sum()

    # Labels
    label_counts = chunk["Is Laundering"].value_counts(dropna=False)

    for label, count in label_counts.items():
        invalid_labels[label] = (
            invalid_labels.get(label, 0) + count
        )

    # Payment formats
    invalid_formats.update(
        set(chunk["Payment Format"].dropna().unique())
        - expected_formats
    )

    # Currencies
    invalid_payment_currencies.update(
        set(chunk["Payment Currency"].dropna().unique())
        - expected_currencies
    )

    invalid_receiving_currencies.update(
        set(chunk["Receiving Currency"].dropna().unique())
        - expected_currencies
    )

print("Total rows:", f"{total_rows:,}")

print("\n=== MISSING VALUES ===")
for col, count in missing_counts.items():
    print(
        f"{col:25} "
        f"{count:,} "
        f"({count / total_rows * 100:.4f}%)"
    )

print("\n=== AMOUNT QUALITY ===")
for key, value in invalid_amounts.items():
    print(f"{key:25} {value:,}")

print("\n=== LABEL VALUES ===")
print(invalid_labels)

print("\n=== INVALID PAYMENT FORMATS ===")
print(invalid_formats)

print("\n=== INVALID PAYMENT CURRENCIES ===")
print(invalid_payment_currencies)

print("\n=== INVALID RECEIVING CURRENCIES ===")
print(invalid_receiving_currencies)

In [ ]:
print("Unique accounts:",
      accounts_df["Account Number"].nunique())

print("Duplicate account numbers:",
      accounts_df["Account Number"].duplicated().sum())

print("Unique banks:",
      accounts_df["Bank ID"].nunique())

print("Unique entities:",
      accounts_df["Entity ID"].nunique())

In [ ]:
duplicate_accounts = accounts_df[
    accounts_df["Account Number"].duplicated(keep=False)
].sort_values("Account Number")

display(duplicate_accounts)

So the account number alone is NOT a globally unique identifier. Instead, the real identity is effectively Bank ID + Account Number

### Create a composite account key

In [ ]:
accounts_df["Account Key"] = (
    accounts_df["Bank ID"].astype(str)
    + "_"
    + accounts_df["Account Number"].astype(str)
)

In [ ]:
print("Unique account keys:",
      accounts_df["Account Key"].nunique())

print("Duplicate account keys:",
      accounts_df["Account Key"].duplicated().sum())

This confirms the account table is structurally clean. So from now on this composite key will be used and also Our transaction file has:

- From Bank: Account
- To Bank: Account.1

So the same key can be constructed for every transaction.

### Transaction → account matching

In [ ]:
account_lookup = set(accounts_df["Account Key"])

total = 0
sender_found = 0
receiver_found = 0

for chunk in pd.read_csv(
    trans_file,
    chunksize=1_000_000,
    usecols=[
        "From Bank",
        "Account",
        "To Bank",
        "Account.1"
    ]
):
    
    # Build composite keys
    sender_key = (
        chunk["From Bank"].astype(str)
        + "_"
        + chunk["Account"].astype(str)
    )

    receiver_key = (
        chunk["To Bank"].astype(str)
        + "_"
        + chunk["Account.1"].astype(str)
    )

    sender_found += sender_key.isin(account_lookup).sum()
    receiver_found += receiver_key.isin(account_lookup).sum()

    total += len(chunk)

print("Total transactions:", f"{total:,}")

print(
    "Sender accounts found:",
    f"{sender_found:,}",
    f"({sender_found / total * 100:.4f}%)"
)

print(
    "Receiver accounts found:",
    f"{receiver_found:,}",
    f"({receiver_found / total * 100:.4f}%)"
)

Every sender and receiver in our 31.9M transactions has corresponding account metadata in the HI-Medium account file. Now a transaction can be determined whether it is between different banks. But different banks doesn't mean different countries. So next we need to build a reliable Bank → Country mapping.

Extract unique banks

In [ ]:
banks_df = accounts_df[
    ["Bank ID", "Bank Name"]
].drop_duplicates()

print("Unique Bank IDs:", banks_df["Bank ID"].nunique())
print("Unique Bank Names:", banks_df["Bank Name"].nunique())

display(banks_df.head(20))

Analyze bank-name patterns

## 5. Country → bank mapping

In [ ]:
# Show banks whose names do NOT contain one of the obvious country names

countries = [
    "Australia", "Brazil", "Canada", "China", "France",
    "Germany", "India", "Israel", "Italy", "Japan",
    "Mexico", "Netherlands", "Portugal", "Russia",
    "Saudi Arabia", "Spain", "Switzerland", "UK",
    "Finland", "Greece", "Ireland", "Austria"
]

country_pattern = "|".join(countries)

unknown_banks = banks_df[
    ~banks_df["Bank Name"].str.contains(
        country_pattern,
        case=False,
        na=False
    )
]

print("Total unique banks:", len(banks_df))
print("Banks with recognizable country:", 
      len(banks_df) - len(unknown_banks))
print("Banks without recognizable country:",
      len(unknown_banks))

display(unknown_banks.head(100))

Investigate the unknown banks

In [ ]:
unknown_counts = (
    unknown_banks["Bank Name"]
    .value_counts()
)

print("Unique unknown bank names:",
      unknown_counts.shape[0])

display(unknown_counts.head(100))

So we have:

- 122,333 Bank IDs
- 75,136 identifiable by country name
- 47,197 not identifiable by country name
- 47,197 unknown Bank-ID records
- 5,591 distinct unknown bank names

Lets see whether the unknown banks are actually associated with particular countries through their transactions.

In [ ]:
unknown_bank_ids = set(unknown_banks["Bank ID"])

unknown_from_count = 0
unknown_to_count = 0

unknown_sender_banks = set()
unknown_receiver_banks = set()

for chunk in pd.read_csv(
    trans_file,
    usecols=["From Bank", "To Bank"],
    chunksize=1_000_000
):

    # Transactions FROM unknown banks
    from_mask = chunk["From Bank"].isin(unknown_bank_ids)

    # Transactions TO unknown banks
    to_mask = chunk["To Bank"].isin(unknown_bank_ids)

    # Add counts
    unknown_from_count += from_mask.sum()
    unknown_to_count += to_mask.sum()

    # Keep unique bank IDs
    unknown_sender_banks.update(
        chunk.loc[from_mask, "From Bank"].unique()
    )

    unknown_receiver_banks.update(
        chunk.loc[to_mask, "To Bank"].unique()
    )


print("Transactions FROM unknown banks:",
      unknown_from_count)

print("Transactions TO unknown banks:",
      unknown_to_count)

print("Unknown banks appearing as senders:",
      len(unknown_sender_banks))

print("Unknown banks appearing as receivers:",
      len(unknown_receiver_banks))

So approximately:

37.0% of all transactions originate from banks whose country isn't obvious from the name and
40.6% terminate at such banks.

Our goal is to determine whether unknown banks actually associated with particular transactions and yes they did. But we don't necessarily need to determine in which country they belong instead we can create bank behavioral features.

### Known vs Unknown bank relationships.

In [ ]:
# Bank IDs whose country we could recognize from the bank name
known_bank_ids = set(
    banks_df.loc[
        ~banks_df["Bank ID"].isin(unknown_bank_ids),
        "Bank ID"
    ]
)

# Counters
known_to_known = 0
known_to_unknown = 0
unknown_to_known = 0
unknown_to_unknown = 0

# Process the 31.9M transactions in chunks
for chunk in pd.read_csv(
    trans_file,
    usecols=["From Bank", "To Bank"],
    chunksize=1_000_000
):

    from_known = chunk["From Bank"].isin(known_bank_ids)
    to_known = chunk["To Bank"].isin(known_bank_ids)

    # Known → Known
    known_to_known += (from_known & to_known).sum()

    # Known → Unknown
    known_to_unknown += (from_known & ~to_known).sum()

    # Unknown → Known
    unknown_to_known += (~from_known & to_known).sum()

    # Unknown → Unknown
    unknown_to_unknown += (~from_known & ~to_known).sum()


# Display results
total = (
    known_to_known
    + known_to_unknown
    + unknown_to_known
    + unknown_to_unknown
)

print("=== BANK COUNTRY KNOWLEDGE CATEGORIES ===")

print(
    f"Known → Known:       {known_to_known:,} "
    f"({known_to_known / total * 100:.2f}%)"
)

print(
    f"Known → Unknown:     {known_to_unknown:,} "
    f"({known_to_unknown / total * 100:.2f}%)"
)

print(
    f"Unknown → Known:     {unknown_to_known:,} "
    f"({unknown_to_known / total * 100:.2f}%)"
)

print(
    f"Unknown → Unknown:   {unknown_to_unknown:,} "
    f"({unknown_to_unknown / total * 100:.2f}%)"
)

print(f"\nTotal checked: {total:,}")

Lets investigate whether Unknown → Unknown transactions are more likely to be laundering

In [ ]:
unknown_bank_ids = set(unknown_banks["Bank ID"])

unknown_unknown_total = 0
unknown_unknown_laundering = 0

unknown_known_total = 0
unknown_known_laundering = 0

known_unknown_total = 0
known_unknown_laundering = 0

known_known_total = 0
known_known_laundering = 0


for chunk in pd.read_csv(
    trans_file,
    usecols=[
        "From Bank",
        "To Bank",
        "Is Laundering"
    ],
    chunksize=1_000_000
):

    from_unknown = chunk["From Bank"].isin(unknown_bank_ids)
    to_unknown = chunk["To Bank"].isin(unknown_bank_ids)

    # Unknown → Unknown
    mask = from_unknown & to_unknown

    unknown_unknown_total += mask.sum()
    unknown_unknown_laundering += (
        chunk.loc[mask, "Is Laundering"] == 1
    ).sum()

    # Unknown → Known
    mask = from_unknown & ~to_unknown

    unknown_known_total += mask.sum()
    unknown_known_laundering += (
        chunk.loc[mask, "Is Laundering"] == 1
    ).sum()

    # Known → Unknown
    mask = ~from_unknown & to_unknown

    known_unknown_total += mask.sum()
    known_unknown_laundering += (
        chunk.loc[mask, "Is Laundering"] == 1
    ).sum()

    # Known → Known
    mask = ~from_unknown & ~to_unknown

    known_known_total += mask.sum()
    known_known_laundering += (
        chunk.loc[mask, "Is Laundering"] == 1
    ).sum()


print("=== LAUNDERING BY BANK KNOWLEDGE ===")

print(
    f"Known → Known: "
    f"{known_known_total:,} total | "
    f"{known_known_laundering:,} laundering | "
    f"{known_known_laundering / known_known_total * 100:.4f}%"
)

print(
    f"Known → Unknown: "
    f"{known_unknown_total:,} total | "
    f"{known_unknown_laundering:,} laundering | "
    f"{known_unknown_laundering / known_unknown_total * 100:.4f}%"
)

print(
    f"Unknown → Known: "
    f"{unknown_known_total:,} total | "
    f"{unknown_known_laundering:,} laundering | "
    f"{unknown_known_laundering / unknown_known_total * 100:.4f}%"
)

print(
    f"Unknown → Unknown: "
    f"{unknown_unknown_total:,} total | "
    f"{unknown_unknown_laundering:,} laundering | "
    f"{unknown_unknown_laundering / unknown_unknown_total * 100:.4f}%"
)

Unknown → Unknown is not automatically suspicious. Even though it represents 33.73% of all transactions, its laundering rate is only 0.0785%.

## 6. Feature Engineering — Clean, Leakage-Aware Pipeline

The final pipeline follows this order:

1. Define the temporal train/validation/test boundaries.
2. Calculate account and relationship statistics using **training data only**.
3. Build transaction-level features.
4. Merge the training-derived sender, receiver, and relationship features.
5. Add bank-knowledge features.
6. Process the raw transaction file in chunks so the complete 31.9M-row dataset is
   never loaded into memory at once.

Before splitting based on time frame we must ballance the laundering effect on the timestamp

In [ ]:
# ============================================================
# ANALYZE LAUNDERING DISTRIBUTION OVER TIME
# ============================================================
CHUNK_SIZE = 1_000_000

# Count transactions and laundering transactions by day
daily_stats = defaultdict(lambda: [0, 0])

for i, chunk in enumerate(
    pd.read_csv(
        trans_file,
        usecols=["Timestamp", "Is Laundering"],
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    chunk["Timestamp"] = pd.to_datetime(chunk["Timestamp"])

    chunk["_date"] = chunk["Timestamp"].dt.date

    grouped = chunk.groupby("_date")["Is Laundering"].agg(
        total="count",
        laundering="sum"
    )

    for date, row in grouped.iterrows():
        daily_stats[date][0] += int(row["total"])
        daily_stats[date][1] += int(row["laundering"])

    print(f"Processed chunk {i}")

# Convert to DataFrame
daily_distribution = pd.DataFrame(
    [
        {
            "Date": date,
            "Total": values[0],
            "Laundering": values[1],
            "Normal": values[0] - values[1],
            "Laundering_Rate": (
                values[1] / values[0] * 100
                if values[0] > 0 else 0
            )
        }
        for date, values in sorted(daily_stats.items())
    ]
)

print("\n" + "=" * 70)
print("DAILY LAUNDERING DISTRIBUTION")
print("=" * 70)

display(daily_distribution)

print("\nTotal transactions:",
      daily_distribution["Total"].sum())

print("Total laundering:",
      daily_distribution["Laundering"].sum())

In [ ]:
# ============================================================
# PLOT LAUNDERING RATE OVER TIME
# ============================================================

plt.figure(figsize=(14, 6))

plt.plot(
    daily_distribution["Date"],
    daily_distribution["Laundering_Rate"],
    marker="o"
)

plt.xlabel("Date")
plt.ylabel("Laundering Rate (%)")
plt.title("Daily Laundering Rate Over Time")

plt.xticks(rotation=45)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The important pattern is:

- Sept 1–16: millions of transactions per day, laundering rate roughly 0.02%–0.25%
- Sept 17 onward: transaction volume suddenly collapses, while laundering remains hundreds/thousands per day, producing ~56–63% laundering
- By Sept 28 there are only 7 transactions, 3 of which are laundering.

We should design the split so that:

- Training contains the large historical period.
- Validation has enough transactions and laundering cases to evaluate the model.
- Test has enough transactions and laundering cases.
- We preserve temporal ordering to avoid leakage.
- We explicitly document the distribution shift.

I would not make the train/validation/test laundering percentages equal because that would destroy the actual temporal structure of this dataset.

So let's use 
| Split          | Period     | Purpose                                                      |
| -------------- | ---------- | ------------------------------------------------------------ |
| **Train**      | Sept 1–16  | Learn normal historical behavior                             |
| **Validation** | Sept 17–20 | Tune/evaluate during the beginning of the distribution shift |
| **Test**       | Sept 21–28 | Final evaluation on later unseen data                        |


This is much better for this project because we're testing whether the model can detect laundering when the transaction behavior changes over time.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CHUNK_SIZE = 1_000_000

# These are the boundaries already established from the dataset.
TRAIN_END = pd.Timestamp("2022-09-16 23:59:59")
VALIDATION_END = pd.Timestamp("2022-09-20 23:59:59")

print("Train end:", TRAIN_END)
print("Validation end:", VALIDATION_END)

### Why the statistics are training-only

Sender, receiver, relationship, and behavioral statistics must not use transactions
from validation or test periods. Otherwise, future information can leak into the
features.

In [ ]:
# ============================================================
# 2. TRAINING-ONLY ACCOUNT STATISTICS
# ============================================================

# Scalar sender statistics
sender_count = {}
sender_total = {}
sender_max = {}
sender_weekend = {}
sender_night = {}
sender_day_mask = {}
sender_hour_mask = {}

# Scalar receiver statistics
receiver_count = {}
receiver_total = {}
receiver_max = {}
receiver_weekend = {}
receiver_night = {}
receiver_day_mask = {}
receiver_hour_mask = {}

chunks_processed = 0
training_rows = 0

for chunk in pd.read_csv(
    trans_file,
    usecols=[
        "Timestamp",
        "Account",
        "Account.1",
        "Amount Received",
        "Amount Paid"
    ],
    chunksize=CHUNK_SIZE
):

    chunks_processed += 1

    chunk["Timestamp"] = pd.to_datetime(chunk["Timestamp"])

    # CSV is time ordered.
    # Once an entire chunk is after TRAIN_END,
    # no later chunk can contain training rows.
    if chunk["Timestamp"].min() > TRAIN_END:
        break

    chunk = chunk[
        chunk["Timestamp"] <= TRAIN_END
    ].copy()

    if chunk.empty:
        continue

    training_rows += len(chunk)

    # --------------------------------------------------------
    # Temporal columns
    # --------------------------------------------------------

    chunk["_day"] = chunk["Timestamp"].dt.day
    chunk["_hour"] = chunk["Timestamp"].dt.hour
    chunk["_dow"] = chunk["Timestamp"].dt.dayofweek

    chunk["_weekend"] = (
        chunk["_dow"] >= 5
    ).astype("int8")

    chunk["_night"] = (
        (chunk["_hour"] < 6) |
        (chunk["_hour"] >= 22)
    ).astype("int8")


    # ========================================================
    # SENDER STATISTICS
    # ========================================================

    g = chunk.groupby("Account", sort=False)

    counts = g.size()
    totals = g["Amount Paid"].sum()
    maximums = g["Amount Paid"].max()
    weekends = g["_weekend"].sum()
    nights = g["_night"].sum()

    # Transaction count
    for account, value in counts.items():
        sender_count[account] = (
            sender_count.get(account, 0) + int(value)
        )

    # Total amount
    for account, value in totals.items():
        sender_total[account] = (
            sender_total.get(account, 0.0) + float(value)
        )

    # Maximum amount
    for account, value in maximums.items():
        sender_max[account] = max(
            sender_max.get(account, 0.0),
            float(value)
        )

    # Weekend transactions
    for account, value in weekends.items():
        sender_weekend[account] = (
            sender_weekend.get(account, 0) + int(value)
        )

    # Night transactions
    for account, value in nights.items():
        sender_night[account] = (
            sender_night.get(account, 0) + int(value)
        )


    # --------------------------------------------------------
    # Sender unique days
    # --------------------------------------------------------

    sender_day_pairs = (
        chunk[["Account", "_day"]]
        .drop_duplicates()
    )

    sender_day_pairs["_bit"] = (
        2 ** (
            sender_day_pairs["_day"].astype("int64") - 1
        )
    )

    sender_day_masks = (
        sender_day_pairs
        .groupby("Account")["_bit"]
        .sum()
    )

    for account, mask in sender_day_masks.items():
        sender_day_mask[account] = (
            sender_day_mask.get(account, 0) |
            int(mask)
        )


    # --------------------------------------------------------
    # Sender unique hours
    # --------------------------------------------------------

    sender_hour_pairs = (
        chunk[["Account", "_hour"]]
        .drop_duplicates()
    )

    sender_hour_pairs["_bit"] = (
        2 ** sender_hour_pairs["_hour"].astype("int64")
    )

    sender_hour_masks = (
        sender_hour_pairs
        .groupby("Account")["_bit"]
        .sum()
    )

    for account, mask in sender_hour_masks.items():
        sender_hour_mask[account] = (
            sender_hour_mask.get(account, 0) |
            int(mask)
        )


    # ========================================================
    # RECEIVER STATISTICS
    # ========================================================

    g = chunk.groupby("Account.1", sort=False)

    counts = g.size()
    totals = g["Amount Received"].sum()
    maximums = g["Amount Received"].max()
    weekends = g["_weekend"].sum()
    nights = g["_night"].sum()

    # Transaction count
    for account, value in counts.items():
        receiver_count[account] = (
            receiver_count.get(account, 0) + int(value)
        )

    # Total amount
    for account, value in totals.items():
        receiver_total[account] = (
            receiver_total.get(account, 0.0) + float(value)
        )

    # Maximum amount
    for account, value in maximums.items():
        receiver_max[account] = max(
            receiver_max.get(account, 0.0),
            float(value)
        )

    # Weekend transactions
    for account, value in weekends.items():
        receiver_weekend[account] = (
            receiver_weekend.get(account, 0) + int(value)
        )

    # Night transactions
    for account, value in nights.items():
        receiver_night[account] = (
            receiver_night.get(account, 0) + int(value)
        )


    # --------------------------------------------------------
    # Receiver unique days
    # --------------------------------------------------------

    receiver_day_pairs = (
        chunk[["Account.1", "_day"]]
        .drop_duplicates()
    )

    receiver_day_pairs["_bit"] = (
        2 ** (
            receiver_day_pairs["_day"].astype("int64") - 1
        )
    )

    receiver_day_masks = (
        receiver_day_pairs
        .groupby("Account.1")["_bit"]
        .sum()
    )

    for account, mask in receiver_day_masks.items():
        receiver_day_mask[account] = (
            receiver_day_mask.get(account, 0) |
            int(mask)
        )


    # --------------------------------------------------------
    # Receiver unique hours
    # --------------------------------------------------------

    receiver_hour_pairs = (
        chunk[["Account.1", "_hour"]]
        .drop_duplicates()
    )

    receiver_hour_pairs["_bit"] = (
        2 ** receiver_hour_pairs["_hour"].astype("int64")
    )

    receiver_hour_masks = (
        receiver_hour_pairs
        .groupby("Account.1")["_bit"]
        .sum()
    )

    for account, mask in receiver_hour_masks.items():
        receiver_hour_mask[account] = (
            receiver_hour_mask.get(account, 0) |
            int(mask)
        )


    print(
        f"Processed training chunk {chunks_processed}"
    )


print("\nFinished.")
print("Training rows:", training_rows)
print("Sender accounts:", len(sender_count))
print("Receiver accounts:", len(receiver_count))

### 3. Build sender and receiver feature tables

The day/hour bit masks are converted into counts with `bit_count()`. This gives the
same meaning as the earlier `nunique()` approach without maintaining a large Python
set for every account.

In [ ]:
# ============================================================
# 3. BUILD SENDER FEATURE TABLE
# ============================================================

sender_features = pd.DataFrame({
    "Account": list(sender_count.keys())
})

sender_features["Sender_Transaction_Count"] = (
    sender_features["Account"].map(sender_count).astype("int64")
)

sender_features["Sender_Total_Amount"] = (
    sender_features["Account"].map(sender_total).astype("float64")
)

sender_features["Sender_Max_Amount"] = (
    sender_features["Account"].map(sender_max).astype("float64")
)

sender_features["Sender_Unique_Receivers"] = 0  # filled from relationship table

sender_features["Sender_Unique_Days"] = (
    sender_features["Account"]
    .map(lambda a: int(sender_day_mask.get(a, 0).bit_count()))
    .astype("int16")
)

sender_features["Sender_Active_Hours"] = (
    sender_features["Account"]
    .map(lambda a: int(sender_hour_mask.get(a, 0).bit_count()))
    .astype("int8")
)

sender_features["Sender_Weekend_Transactions"] = (
    sender_features["Account"]
    .map(sender_weekend)
    .fillna(0)
    .astype("int64")
)

sender_features["Sender_Night_Transactions"] = (
    sender_features["Account"]
    .map(sender_night)
    .fillna(0)
    .astype("int64")
)

sender_features["Sender_Average_Amount"] = (
    sender_features["Sender_Total_Amount"]
    / sender_features["Sender_Transaction_Count"]
)

sender_features["Sender_Average_Daily_Transactions"] = (
    sender_features["Sender_Transaction_Count"]
    / sender_features["Sender_Unique_Days"].replace(0, np.nan)
).fillna(0)

print("Sender feature table:", sender_features.shape)
display(sender_features.head())

In [ ]:
# ============================================================
# BUILD RECEIVER FEATURE TABLE
# ============================================================

receiver_features = pd.DataFrame({
    "Account": list(receiver_count.keys())
})

receiver_features["Receiver_Transaction_Count"] = (
    receiver_features["Account"].map(receiver_count).astype("int64")
)

receiver_features["Receiver_Total_Amount"] = (
    receiver_features["Account"].map(receiver_total).astype("float64")
)

receiver_features["Receiver_Max_Amount"] = (
    receiver_features["Account"].map(receiver_max).astype("float64")
)

receiver_features["Receiver_Unique_Senders"] = 0  # filled from relationship table

receiver_features["Receiver_Unique_Days"] = (
    receiver_features["Account"]
    .map(lambda a: int(receiver_day_mask.get(a, 0).bit_count()))
    .astype("int16")
)

receiver_features["Receiver_Active_Hours"] = (
    receiver_features["Account"]
    .map(lambda a: int(receiver_hour_mask.get(a, 0).bit_count()))
    .astype("int8")
)

receiver_features["Receiver_Weekend_Transactions"] = (
    receiver_features["Account"]
    .map(receiver_weekend)
    .fillna(0)
    .astype("int64")
)

receiver_features["Receiver_Night_Transactions"] = (
    receiver_features["Account"]
    .map(receiver_night)
    .fillna(0)
    .astype("int64")
)

receiver_features["Receiver_Average_Amount"] = (
    receiver_features["Receiver_Total_Amount"]
    / receiver_features["Receiver_Transaction_Count"]
)

receiver_features["Receiver_Average_Daily_Transactions"] = (
    receiver_features["Receiver_Transaction_Count"]
    / receiver_features["Receiver_Unique_Days"].replace(0, np.nan)
).fillna(0)

print("Receiver feature table:", receiver_features.shape)
display(receiver_features.head())

## 7. Training-only relationship statistics

This is the part that previously caused the notebook to run out of memory.

Instead of keeping the grouped results from all 32 chunks in one huge list, the code
below periodically combines partial relationship tables. This keeps the peak memory
much lower while preserving the final global sender → receiver statistics.

In [ ]:
# ============================================================
# 5. TRAINING RELATIONSHIP STATISTICS
# ============================================================

relationship_parts = []
relationship_batch_size = 4
chunks_processed = 0

def combine_relationship_parts(parts):
    combined = pd.concat(parts, ignore_index=True)

    combined = (
        combined
        .groupby(["Account", "Account.1"], as_index=False)
        .agg(
            Relationship_Transaction_Count=(
                "Relationship_Transaction_Count", "sum"
            ),
            Relationship_Total_Amount=(
                "Relationship_Total_Amount", "sum"
            ),
            Relationship_Max_Amount=(
                "Relationship_Max_Amount", "max"
            )
        )
    )

    return combined

for chunk in pd.read_csv(
    trans_file,
    usecols=[
        "Timestamp",
        "Account",
        "Account.1",
        "Amount Received"
    ],
    chunksize=CHUNK_SIZE
):
    chunks_processed += 1

    chunk["Timestamp"] = pd.to_datetime(chunk["Timestamp"])

    if chunk["Timestamp"].min() > TRAIN_END:
        break

    chunk = chunk[chunk["Timestamp"] <= TRAIN_END]

    if chunk.empty:
        continue

    part = (
        chunk
        .groupby(["Account", "Account.1"], sort=False)
        .agg(
            Relationship_Transaction_Count=(
                "Amount Received", "count"
            ),
            Relationship_Total_Amount=(
                "Amount Received", "sum"
            ),
            Relationship_Max_Amount=(
                "Amount Received", "max"
            )
        )
        .reset_index()
    )

    relationship_parts.append(part)

    # Periodically reduce the partial tables.
    if len(relationship_parts) >= relationship_batch_size:
        relationship_parts = [
            combine_relationship_parts(relationship_parts)
        ]

    print(f"Processed relationship chunk {chunks_processed}")

relationship_stats = combine_relationship_parts(
    relationship_parts
)

relationship_stats["Relationship_Average_Amount"] = (
    relationship_stats["Relationship_Total_Amount"]
    / relationship_stats["Relationship_Transaction_Count"]
)

print("\nTraining relationship statistics:", relationship_stats.shape)
display(relationship_stats.head())

# We no longer need the partial relationship tables.
del relationship_parts

In [ ]:
# ============================================================
# 6. COMPLETE UNIQUE-COUNTER FEATURES FROM RELATIONSHIPS
# ============================================================

sender_unique_receivers = (
    relationship_stats
    .groupby("Account")["Account.1"]
    .nunique()
)

receiver_unique_senders = (
    relationship_stats
    .groupby("Account.1")["Account"]
    .nunique()
)

sender_features["Sender_Unique_Receivers"] = (
    sender_features["Account"]
    .map(sender_unique_receivers)
    .fillna(0)
    .astype("int64")
)

receiver_features["Receiver_Unique_Senders"] = (
    receiver_features["Account"]
    .map(receiver_unique_senders)
    .fillna(0)
    .astype("int64")
)

print("Sender features:", sender_features.shape)
print("Receiver features:", receiver_features.shape)

## 8. Transaction-level feature function

These features describe the individual transaction itself. They do not require
historical aggregation tables.

In [ ]:
# ============================================================
# 7. TRANSACTION-LEVEL FEATURES
# ============================================================

def add_transaction_features(df):
    df = df.copy()

    df["Timestamp"] = pd.to_datetime(df["Timestamp"])

    df["Hour"] = df["Timestamp"].dt.hour.astype("int8")
    df["Day"] = df["Timestamp"].dt.day.astype("int8")
    df["DayOfWeek"] = df["Timestamp"].dt.dayofweek.astype("int8")

    df["IsWeekend"] = (
        df["DayOfWeek"] >= 5
    ).astype("int8")

    df["IsNight"] = (
        (df["Hour"] < 6) |
        (df["Hour"] >= 22)
    ).astype("int8")

    df["Amount_Difference"] = (
        df["Amount Received"] - df["Amount Paid"]
    )

    df["Amount_Absolute_Difference"] = (
        df["Amount_Difference"].abs()
    )

    df["Amount_Ratio"] = (
        df["Amount Received"]
        / df["Amount Paid"].replace(0, np.nan)
    )

    df["Amount_Log"] = np.log1p(
        df["Amount Received"].clip(lower=0)
    )

    df["Currency_Match"] = (
        df["Receiving Currency"]
        == df["Payment Currency"]
    ).astype("int8")

    df["Same_Account"] = (
        df["Account"] == df["Account.1"]
    ).astype("int8")

    df["Same_Bank"] = (
        df["From Bank"] == df["To Bank"]
    ).astype("int8")

    return df

## 9. Bank-knowledge features

The existing bank-name analysis from the notebook is retained. We use the
`unknown_bank_ids` already created earlier in the notebook.

In [ ]:
# ============================================================
# 8. BANK KNOWLEDGE
# ============================================================

def add_bank_knowledge_features(df):
    df = df.copy()

    from_unknown = df["From Bank"].isin(unknown_bank_ids)
    to_unknown = df["To Bank"].isin(unknown_bank_ids)

    df["Bank_Knowledge"] = "Known_Known"

    df.loc[
        from_unknown & ~to_unknown,
        "Bank_Knowledge"
    ] = "Unknown_Known"

    df.loc[
        ~from_unknown & to_unknown,
        "Bank_Knowledge"
    ] = "Known_Unknown"

    df.loc[
        from_unknown & to_unknown,
        "Bank_Knowledge"
    ] = "Unknown_Unknown"

    df["Unknown_Bank_Involved"] = (
        from_unknown | to_unknown
    ).astype("int8")

    return df

## 10. Final chunk feature engineering

All historical aggregate features come from the training period only.

If an account or sender → receiver relationship did not exist during training,
its historical aggregate features are filled with zero. This is important because
new accounts/relationships can appear in validation or test.

In [ ]:
# ============================================================
# 9. FINAL CHUNK ENGINEERING FUNCTION
# ============================================================

sender_merge = sender_features.copy()

receiver_merge = receiver_features.rename(
    columns={"Account": "Account.1"}
)

relationship_merge = relationship_stats.copy()


SENDER_FEATURE_COLUMNS = [
    "Sender_Transaction_Count",
    "Sender_Total_Amount",
    "Sender_Max_Amount",
    "Sender_Average_Amount",
    "Sender_Unique_Receivers",
    "Sender_Unique_Days",
    "Sender_Active_Hours",
    "Sender_Weekend_Transactions",
    "Sender_Night_Transactions",
    "Sender_Average_Daily_Transactions",
]

RECEIVER_FEATURE_COLUMNS = [
    "Receiver_Transaction_Count",
    "Receiver_Total_Amount",
    "Receiver_Max_Amount",
    "Receiver_Average_Amount",
    "Receiver_Unique_Senders",
    "Receiver_Unique_Days",
    "Receiver_Active_Hours",
    "Receiver_Weekend_Transactions",
    "Receiver_Night_Transactions",
    "Receiver_Average_Daily_Transactions",
]

RELATIONSHIP_FEATURE_COLUMNS = [
    "Relationship_Transaction_Count",
    "Relationship_Total_Amount",
    "Relationship_Max_Amount",
    "Relationship_Average_Amount",
]


def engineer_chunk(chunk):
    df = add_transaction_features(chunk)

    # Sender history
    df = df.merge(
        sender_merge,
        on="Account",
        how="left"
    )

    # Receiver history
    df = df.merge(
        receiver_merge,
        on="Account.1",
        how="left"
    )

    # Sender -> receiver history
    df = df.merge(
        relationship_merge,
        on=["Account", "Account.1"],
        how="left"
    )

    # New accounts / relationships have no training history.
    historical_columns = (
        SENDER_FEATURE_COLUMNS
        + RECEIVER_FEATURE_COLUMNS
        + RELATIONSHIP_FEATURE_COLUMNS
    )

    df[historical_columns] = (
        df[historical_columns].fillna(0)
    )

    # Bank knowledge
    df = add_bank_knowledge_features(df)

    return df

In [ ]:
# ============================================================
# 10. TEST THE CLEAN PIPELINE ON ONE CHUNK
# ============================================================

test_chunk = pd.read_csv(
    trans_file,
    nrows=10_000
)

test_features = engineer_chunk(test_chunk)

print("Original shape:", test_chunk.shape)
print("Engineered shape:", test_features.shape)

print("\nMissing values:")
print(
    test_features.isna().sum()[
        test_features.isna().sum() > 0
    ]
)

print("\nColumns:")
for i, col in enumerate(test_features.columns, 1):
    print(f"{i:02d}. {col}")

display(test_features.head())

del test_chunk
del test_features

## 11. Create train / validation / test feature files

This final pass processes the raw 31.9M transactions in chunks.

It does **not** concatenate all transactions into one DataFrame. Each processed split
is written to disk separately.

The resulting files can later be read in chunks for preprocessing and model training.

In [ ]:
# ============================================================
# 11. CHUNKED TRAIN / VALIDATION / TEST FEATURE CREATION
# ============================================================

output_dir = "/kaggle/working/aml_features"
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------
# Remove old generated feature files
# ------------------------------------------------------------

for filename in os.listdir(output_dir):
    if filename.endswith(".csv"):
        os.remove(
            os.path.join(output_dir, filename)
        )

# ------------------------------------------------------------
# Counters
# ------------------------------------------------------------

split_counters = {
    "train": 0,
    "validation": 0,
    "test": 0
}

split_rows = {
    "train": 0,
    "validation": 0,
    "test": 0
}

# ------------------------------------------------------------
# Raw columns
# ------------------------------------------------------------

raw_usecols = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering"
]

# ------------------------------------------------------------
# Process dataset chunk by chunk
# ------------------------------------------------------------

for i, chunk in enumerate(
    pd.read_csv(
        trans_file,
        usecols=raw_usecols,
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    # --------------------------------------------------------
    # Convert timestamp BEFORE engineering
    # --------------------------------------------------------

    chunk["Timestamp"] = pd.to_datetime(
        chunk["Timestamp"]
    )

    original_rows = len(chunk)

    # --------------------------------------------------------
    # Engineer features
    # --------------------------------------------------------

    features = engineer_chunk(chunk)

    engineered_rows = len(features)

    # --------------------------------------------------------
    # IMPORTANT: verify no rows disappeared
    # --------------------------------------------------------

    if engineered_rows != original_rows:

        raise ValueError(
            f"ROW LOSS DETECTED in chunk {i}: "
            f"original={original_rows:,}, "
            f"engineered={engineered_rows:,}"
        )

    # --------------------------------------------------------
    # Split by timestamp
    # --------------------------------------------------------

    train_mask = (
        features["Timestamp"] <= TRAIN_END
    )

    validation_mask = (
        (features["Timestamp"] > TRAIN_END)
        &
        (features["Timestamp"] <= VALIDATION_END)
    )

    test_mask = (
        features["Timestamp"] > VALIDATION_END
    )

    train_part = features.loc[
        train_mask
    ].copy()

    validation_part = features.loc[
        validation_mask
    ].copy()

    test_part = features.loc[
        test_mask
    ].copy()

    # --------------------------------------------------------
    # Store row counts
    # --------------------------------------------------------

    split_rows["train"] += len(train_part)
    split_rows["validation"] += len(validation_part)
    split_rows["test"] += len(test_part)

    # --------------------------------------------------------
    # Verify every engineered row belongs to exactly one split
    # --------------------------------------------------------

    total_split_rows = (
        len(train_part)
        + len(validation_part)
        + len(test_part)
    )

    if total_split_rows != engineered_rows:

        raise ValueError(
            f"SPLIT ERROR in chunk {i}: "
            f"engineered={engineered_rows:,}, "
            f"split={total_split_rows:,}"
        )

    # --------------------------------------------------------
    # Save each split
    # --------------------------------------------------------

    split_parts = {
        "train": train_part,
        "validation": validation_part,
        "test": test_part
    }

    for split_name, part in split_parts.items():

        if part.empty:
            continue

        split_counters[split_name] += 1

        file_path = os.path.join(
            output_dir,
            f"{split_name}_features_"
            f"{split_counters[split_name]:02d}.csv"
        )

        part.to_csv(
            file_path,
            index=False
        )

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    print(
        f"Processed chunk {i}: "
        f"train={len(train_part):,}, "
        f"validation={len(validation_part):,}, "
        f"test={len(test_part):,}"
    )

    del chunk
    del features
    del train_part
    del validation_part
    del test_part


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("FINISHED")
print("=" * 60)

print("\nFiles created:")

for split_name in [
    "train",
    "validation",
    "test"
]:
    print(
        f"{split_name}: "
        f"{split_counters[split_name]} files"
    )

print("\nRows created:")

for split_name in [
    "train",
    "validation",
    "test"
]:
    print(
        f"{split_name}: "
        f"{split_rows[split_name]:,}"
    )

total_rows = sum(
    split_rows.values()
)

print("\nTotal rows:", f"{total_rows:,}")

print(
    "Output directory:",
    output_dir
)

Now let's calculate the actual number and percentage of laundering transactions in our train, validation, and test files. That will tell us whether the current time split is already good enough or not.

In [ ]:
# ============================================================
# 12. CHECK TARGET DISTRIBUTION FOR ALL SPLITS
# ============================================================

def check_split_distribution(split_name):

    files = sorted([
        f for f in os.listdir(output_dir)
        if f.startswith(split_name + "_features_")
        and f.endswith(".csv")
    ])

    total_rows = 0
    laundering_rows = 0
    normal_rows = 0

    for filename in files:

        file_path = os.path.join(
            output_dir,
            filename
        )

        target = pd.read_csv(
            file_path,
            usecols=["Is Laundering"]
        )["Is Laundering"]

        laundering = int(target.sum())
        rows = len(target)

        laundering_rows += laundering
        normal_rows += rows - laundering
        total_rows += rows

    print("\n" + "=" * 50)
    print(split_name.upper())
    print("=" * 50)

    print("Files:", len(files))
    print("Total transactions:", total_rows)
    print("Normal transactions:", normal_rows)
    print("Laundering transactions:", laundering_rows)

    if total_rows > 0:
        print(
            "Laundering percentage:",
            laundering_rows / total_rows * 100
        )

        print(
            "Normal percentage:",
            normal_rows / total_rows * 100
        )


# Check all three splits
check_split_distribution("train")
check_split_distribution("validation")
check_split_distribution("test")

This now our current valid split for our project

| Split      | Transactions |     Normal | Laundering | Laundering % |
| ---------- | -----------: | ---------: | ---------: | -----------: |
| Train      |   31,891,251 | 31,860,105 |     31,146 |   **0.098%** |
| Validation |        5,661 |      2,359 |      3,302 |   **58.33%** |
| Test       |        1,326 |        544 |        782 |   **58.97%** |


### Important modeling note

At this point the feature-engineering stage is complete.

The account and relationship statistics were calculated from the training period only,
which prevents future validation/test transactions from contributing historical
information to those features.

In [ ]:
# ============================================================
# 14. MODEL FEATURE COLUMNS
# ============================================================

MODEL_NUMERIC_FEATURES = [
    "Amount Received",
    "Amount Paid",

    "Sender_Transaction_Count",
    "Sender_Total_Amount",
    "Sender_Max_Amount",
    "Sender_Average_Amount",
    "Sender_Unique_Receivers",
    "Sender_Unique_Days",
    "Sender_Active_Hours",
    "Sender_Weekend_Transactions",
    "Sender_Night_Transactions",
    "Sender_Average_Daily_Transactions",

    "Receiver_Transaction_Count",
    "Receiver_Total_Amount",
    "Receiver_Max_Amount",
    "Receiver_Average_Amount",
    "Receiver_Unique_Senders",
    "Receiver_Unique_Days",
    "Receiver_Active_Hours",
    "Receiver_Weekend_Transactions",
    "Receiver_Night_Transactions",
    "Receiver_Average_Daily_Transactions",

    "Relationship_Transaction_Count",
    "Relationship_Total_Amount",
    "Relationship_Max_Amount",
    "Relationship_Average_Amount",

    "Hour",
    "Day",
    "DayOfWeek",
    "IsWeekend",
    "IsNight",
    "Amount_Difference",
    "Amount_Absolute_Difference",
    "Amount_Ratio",
    "Amount_Log",
    "Currency_Match",
    "Same_Account",
    "Same_Bank",
    "Unknown_Bank_Involved",
]

MODEL_CATEGORICAL_FEATURES = [
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
    "Bank_Knowledge",
]

TARGET_COLUMN = "Is Laundering"

print("Numeric features:", len(MODEL_NUMERIC_FEATURES))
print("Categorical features:", len(MODEL_CATEGORICAL_FEATURES))
print("Target:", TARGET_COLUMN)

## 12. Imports and training configuration

In [ ]:
# Paths

output_dir = "/kaggle/working/aml_features"

train_files = sorted(
    glob.glob(os.path.join(output_dir, "train_features_*.csv"))
)

validation_files = sorted(
    glob.glob(os.path.join(output_dir, "validation_features_*.csv"))
)

test_files = sorted(
    glob.glob(os.path.join(output_dir, "test_features_*.csv"))
)

print("Training files:", len(train_files))
print("Validation files:", len(validation_files))
print("Test files:", len(test_files))

# ------------------------------------------------------------
# Training sampling configuration
# ------------------------------------------------------------

# Keep every laundering transaction.
# For every laundering transaction, keep this many
# normal transactions.

NEGATIVE_TO_POSITIVE_RATIO = 10

print("\nNegative : Positive ratio:", 
      f"{NEGATIVE_TO_POSITIVE_RATIO}:1")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("Random state:", RANDOM_STATE)
print("\nConfiguration ready.")

Since you want to compare 7 different models, we should first build a reusable preprocessing/training setup that handles your 39 numeric + 4 categorical features without loading all 31.9M rows into RAM.

In [ ]:
# LOAD ONE TRAINING CHUNK

# Use the first training file for model development.
# We will later train the final models using all training files.
train_file = os.path.join(
    output_dir,
    "train_features_01.csv"
)

train_sample = pd.read_csv(train_file)

print("Training sample shape:", train_sample.shape)

# ------------------------------------------------------------
# Separate features and target
# ------------------------------------------------------------

X_sample = train_sample[
    MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
].copy()

y_sample = train_sample[TARGET_COLUMN].copy()

print("X shape:", X_sample.shape)
print("y shape:", y_sample.shape)

print("\nTarget distribution:")
print(y_sample.value_counts())

print("\nTarget percentages:")
print(
    y_sample.value_counts(normalize=True) * 100
)

print("\nMissing values:")
print(X_sample.isna().sum().sum())

In [ ]:
# ============================================================
# BUILD MODEL TRAINING SAMPLE
# ============================================================

training_parts = []

total_laundering = 0
total_normal_seen = 0

for i, file_path in enumerate(train_files, start=1):

    print(f"Processing training file {i}/{len(train_files)}")

    df = pd.read_csv(file_path)

    # Separate laundering and normal transactions
    laundering = df[
        df[TARGET_COLUMN] == 1
    ]

    normal = df[
        df[TARGET_COLUMN] == 0
    ]

    total_laundering += len(laundering)
    total_normal_seen += len(normal)

    # Keep ALL laundering transactions

    if len(laundering) > 0:
        training_parts.append(laundering)

        # Sample normal transactions
        n_normal = min(
            len(normal),
            len(laundering) * NEGATIVE_TO_POSITIVE_RATIO
        )

        if n_normal > 0:

            normal_sample = normal.sample(
                n=n_normal,
                random_state=RANDOM_STATE
            )

            training_parts.append(normal_sample)

    del df
    del laundering
    del normal

# Combine sampled data
train_model_df = pd.concat(
    training_parts,
    ignore_index=True
)

# Shuffle the final training data
train_model_df = train_model_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("\n" + "=" * 60)
print("TRAINING SAMPLE CREATED")
print("=" * 60)

print("Total laundering transactions available:",
      total_laundering)

print("Total normal transactions available:",
      total_normal_seen)

print("Final training shape:",
      train_model_df.shape)

print("\nTarget distribution:")
print(
    train_model_df[TARGET_COLUMN].value_counts()
)

print("\nTarget percentages:")
print(
    train_model_df[TARGET_COLUMN]
    .value_counts(normalize=True) * 100
)

In [ ]:
# ============================================================
# FINALIZE TRAINING FEATURES AND TARGET
# ============================================================

X_train = train_model_df[
    MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
].copy()

y_train = train_model_df[
    TARGET_COLUMN
].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nNumeric feature count:",
      len(MODEL_NUMERIC_FEATURES))

print("Categorical feature count:",
      len(MODEL_CATEGORICAL_FEATURES))

print("\nTarget distribution:")
print(y_train.value_counts())

print("\nMissing values in X_train:",
      X_train.isna().sum().sum())

In [ ]:
# Numeric preprocessing

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# Combine both

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            MODEL_NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_transformer,
            MODEL_CATEGORICAL_FEATURES
        )
    ]
)

print("Preprocessor created successfully.")

print("\nNumeric features:",
      len(MODEL_NUMERIC_FEATURES))

print("Categorical features:",
      len(MODEL_CATEGORICAL_FEATURES))

print("\nPipeline:")
print(preprocessor)

In [ ]:
# ============================================================
# FIT PREPROCESSOR ON TRAINING DATA
# ============================================================

print("Fitting preprocessor on training data...")

X_train_processed = preprocessor.fit_transform(X_train)

# Get the feature names created by the preprocessor
PROCESSED_FEATURE_NAMES = (
    preprocessor.get_feature_names_out()
)

# Convert processed training data to DataFrame
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=PROCESSED_FEATURE_NAMES,
    index=X_train.index
)

print("\nPreprocessing complete.")

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Processed data type:",
      type(X_train_processed))

print("Number of feature names:",
      len(PROCESSED_FEATURE_NAMES))

print("\nFirst 10 feature names:")
print(
    PROCESSED_FEATURE_NAMES[:10]
)

In [ ]:
# ============================================================
# PREPROCESS VALIDATION DATA
# ============================================================

print("=" * 60)
print("PREPROCESSING VALIDATION DATA")
print("=" * 60)

# Transform using the already-fitted training preprocessor
X_validation_processed = preprocessor.transform(
    X_validation
)

# Convert to DataFrame with the SAME feature names as training
X_validation_processed = pd.DataFrame(
    X_validation_processed,
    columns=PROCESSED_FEATURE_NAMES,
    index=X_validation.index
)

print("\nValidation preprocessing complete.")

print("Original validation shape:",
      X_validation.shape)

print("Processed validation shape:",
      X_validation_processed.shape)

print("Processed data type:",
      type(X_validation_processed))

print("Number of feature names:",
      len(X_validation_processed.columns))

print("\nFirst 10 feature names:")
print(
    X_validation_processed.columns[:10].tolist()
)

In [ ]:
def evaluate_model(
    model,
    X,
    y,
    model_name="Model",
    threshold=0.50,
    return_predictions=False
):
    """
    Evaluate a binary classification model using a common
    set of AML-focused metrics.
    """

    # Probability predictions

    y_probability = model.predict_proba(X)[:, 1]

    # Apply classification threshold

    y_prediction = (
        y_probability >= threshold
    ).astype(int)

    # Calculate metrics

    precision = precision_score(
        y,
        y_prediction,
        zero_division=0
    )

    recall = recall_score(
        y,
        y_prediction,
        zero_division=0
    )

    f1 = f1_score(
        y,
        y_prediction,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y,
        y_probability
    )

    pr_auc = average_precision_score(
        y,
        y_probability
    )

    tn, fp, fn, tp = confusion_matrix(
        y,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    # Store results

    results = {
        "Model": model_name,
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "True_Negatives": tn,
        "False_Positives": fp,
        "False_Negatives": fn,
        "True_Positives": tp
    }

    if return_predictions:
        return results, y_prediction, y_probability

    return results


print("Common evaluation function created.")

In [ ]:
# ============================================================
# LOAD VALIDATION DATA FOR TIME-BASED K-FOLD
# ============================================================

validation_parts = []

for i, file_path in enumerate(
    validation_files,
    start=1
):
    print(
        f"Loading validation file "
        f"{i}/{len(validation_files)}"
    )

    df = pd.read_csv(file_path)

    validation_parts.append(df)

# ------------------------------------------------------------
# Combine validation files
# ------------------------------------------------------------

validation_df = pd.concat(
    validation_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

validation_df["Timestamp"] = pd.to_datetime(
    validation_df["Timestamp"]
)

validation_df = validation_df.sort_values(
    "Timestamp"
).reset_index(drop=True)

In [ ]:
# COMMON CHRONOLOGICAL K-FOLD MODEL EVALUATION

def evaluate_model_chronological_cv(
    model,
    model_name,
    X_train_processed,
    y_train,
    validation_folds,
    preprocessor,
    threshold=0.50
):
    """
    Train one model on the training set and evaluate it across
    chronological validation folds.

    The model is trained only on the training data.
    Validation folds are used only for evaluation.
    """

    print("\n" + "=" * 70)
    print(f"{model_name}")
    print("=" * 70)

    # Train model

    print("Training model...")

    model.fit(
        X_train_processed,
        y_train
    )

    print("Training complete.")

    fold_results = []

    # Evaluate every chronological fold

    for fold_number, fold_df in enumerate(
        validation_folds,
        start=1
    ):

        print(
            f"\nEvaluating Fold {fold_number}/"
            f"{len(validation_folds)}..."
        )

        # Features and target

        X_fold = fold_df[
            MODEL_NUMERIC_FEATURES
            + MODEL_CATEGORICAL_FEATURES
        ]

        y_fold = fold_df[
            TARGET_COLUMN
        ]

        # Use the training-fitted preprocessor.
        # Never fit it on validation data.

        X_fold_processed = pd.DataFrame(
            preprocessor.transform(X_fold),
            columns=PROCESSED_FEATURE_NAMES,
            index=X_fold.index
        )

        # Common evaluation function

        result = evaluate_model(
            model=model,
            X=X_fold_processed,
            y=y_fold,
            model_name=model_name,
            threshold=threshold
        )

        result["Fold"] = fold_number
        result["Validation_Rows"] = len(fold_df)
        result["Validation_Laundering"] = int(
            y_fold.sum()
        )

        fold_results.append(result)

        print(
            f"Precision: {result['Precision']:.4f} | "
            f"Recall: {result['Recall']:.4f} | "
            f"F1: {result['F1']:.4f} | "
            f"ROC-AUC: {result['ROC_AUC']:.4f} | "
            f"PR-AUC: {result['PR_AUC']:.4f}"
        )

    # Convert fold results to DataFrame

    fold_results_df = pd.DataFrame(
        fold_results
    )

    # Calculate mean and standard deviation

    metric_columns = [
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "PR_AUC"
    ]

    summary = {
        "Model": model_name
    }

    for metric in metric_columns:

        summary[f"{metric}_Mean"] = (
            fold_results_df[metric].mean()
        )

        summary[f"{metric}_Std"] = (
            fold_results_df[metric].std()
        )

    print("\n" + "-" * 70)
    print(f"{model_name} — CROSS-VALIDATION SUMMARY")
    print("-" * 70)

    for metric in metric_columns:

        mean = summary[f"{metric}_Mean"]
        std = summary[f"{metric}_Std"]

        print(
            f"{metric}: "
            f"{mean:.4f} ± {std:.4f}"
        )

    return model, fold_results_df, summary


print(
    "Common chronological K-fold evaluation function created."
)

In [ ]:
# ============================================================
# CREATE 5 CHRONOLOGICAL VALIDATION FOLDS
# ============================================================

N_FOLDS = 5

validation_folds = []

fold_size = len(validation_df) // N_FOLDS

for i in range(N_FOLDS):

    start = i * fold_size

    # Make the final fold include any remaining rows
    if i == N_FOLDS - 1:
        end = len(validation_df)
    else:
        end = (i + 1) * fold_size

    fold_df = validation_df.iloc[
        start:end
    ].copy()

    validation_folds.append(fold_df)

    print(
        f"Fold {i + 1}: "
        f"rows={len(fold_df):,} | "
        f"laundering={fold_df[TARGET_COLUMN].sum():,} | "
        f"normal={(fold_df[TARGET_COLUMN] == 0).sum():,} | "
        f"start={fold_df['Timestamp'].min()} | "
        f"end={fold_df['Timestamp'].max()}"
    )

print("\n" + "=" * 70)
print("CHRONOLOGICAL VALIDATION FOLDS CREATED")
print("=" * 70)
print("Number of folds:", len(validation_folds))
print("Total validation rows:",
      sum(len(fold) for fold in validation_folds))

In [ ]:
# ============================================================
# DEFINE 7 MODELS
# ============================================================

# Class imbalance

NEGATIVE_POSITIVE_RATIO = 10

# ------------------------------------------------------------
# 1. Logistic Regression
# ------------------------------------------------------------

models = {}

models["Logistic Regression"] = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 2. Random Forest
# ------------------------------------------------------------

models["Random Forest"] = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 3. Extra Trees
# ------------------------------------------------------------

models["Extra Trees"] = ExtraTreesClassifier(
    n_estimators=150,
    class_weight="balanced",
    max_depth=15,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 4. HistGradientBoosting
# ------------------------------------------------------------

models["HistGradientBoosting"] = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 5. XGBoost
# ------------------------------------------------------------

from xgboost import XGBClassifier

models["XGBoost"] = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=NEGATIVE_POSITIVE_RATIO,
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 6. LightGBM
# ------------------------------------------------------------

from lightgbm import LGBMClassifier

models["LightGBM"] = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.08,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=NEGATIVE_POSITIVE_RATIO,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1
)

# ------------------------------------------------------------
# 7. CatBoost
# ------------------------------------------------------------

from catboost import CatBoostClassifier

models["CatBoost"] = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.08,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    verbose=False,
    random_seed=RANDOM_STATE
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 60)
print("7 MODELS READY")
print("=" * 60)

for i, name in enumerate(models.keys(), start=1):
    print(f"{i}. {name}")

print("=" * 60)

Now lets train the 7 models

In [ ]:
# TRAIN ALL 7 MODELS

trained_models = {}

print("=" * 70)
print("TRAINING 7 MODELS")
print("=" * 70)

for model_name, model in models.items():

    print("\n" + "-" * 70)
    print(f"Training: {model_name}")
    print("-" * 70)

    model.fit(
        X_train_processed,
        y_train
    )

    trained_models[model_name] = model

    print(f"{model_name} training complete.")

print("\n" + "=" * 70)
print("ALL 7 MODELS TRAINED SUCCESSFULLY")
print("=" * 70)

print("\nTrained models:")
for i, model_name in enumerate(
    trained_models.keys(),
    start=1
):
    print(f"{i}. {model_name}")

Now lets do normal evaluation for each model

In [ ]:
# ============================================================
# EVALUATE ALL 7 TRAINED MODELS
# ============================================================

validation_results = []
validation_details = {}

print("=" * 70)
print("EVALUATING ALL 7 MODELS ON VALIDATION SET")
print("=" * 70)

for model_name, model in trained_models.items():

    print("\n" + "-" * 70)
    print(f"Evaluating: {model_name}")
    print("-" * 70)

    result = evaluate_model(
        model=model,
        X=X_validation_processed,
        y=y_validation,
        model_name=model_name
    )

    validation_results.append(result)
    validation_details[model_name] = result

print("\n" + "=" * 70)
print("ALL 7 MODELS EVALUATED")
print("=" * 70)

validation_comparison = pd.DataFrame(
    validation_results
)

display(validation_comparison)

Random Forest is currently the best overall candidate. However, HistGradientBoosting has the highest PR-AUC: 96.15%, which is particularly important for our highly imbalanced AML problem. So we shouldn't simply say "Random Forest wins" yet.

### Why we need the K-fold validation

Our validation set has an unusual distribution, that's roughly 58% laundering, while the original training data is extremely imbalanced. Therefore, these validation metrics are useful, but we want to know "Do these models remain strong across different chronological portions of the validation period?"

In [ ]:
# ============================================================
# RUN CHRONOLOGICAL K-FOLD VALIDATION FOR ALL 7 MODELS
# ============================================================

all_cv_results = {}
all_cv_summaries = []

print("\n" + "=" * 70)
print("CHRONOLOGICAL K-FOLD VALIDATION — ALL 7 MODELS")
print("=" * 70)

for model_name, model in models.items():

    trained_model, fold_results, summary = (
        evaluate_model_chronological_cv(
            model=model,
            model_name=model_name,
            X_train_processed=X_train_processed,
            y_train=y_train,
            validation_folds=validation_folds,
            preprocessor=preprocessor,
            threshold=0.50
        )
    )

    # Store detailed fold results
    all_cv_results[model_name] = fold_results

    # Store summary
    all_cv_summaries.append(summary)


# ============================================================
# COMBINED SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(
    all_cv_summaries
)

print("\n" + "=" * 70)
print("ALL 7 MODELS — CROSS-VALIDATION SUMMARY")
print("=" * 70)

display(
    cv_summary_df.sort_values(
        by="PR_AUC_Mean",
        ascending=False
    ).reset_index(drop=True)
)

In [ ]:
# ============================================================
# VISUALIZE K-FOLD CROSS-VALIDATION RESULTS
# ============================================================

# Sort models by PR-AUC mean
plot_df = cv_summary_df.sort_values(
    by="PR_AUC_Mean",
    ascending=False
).reset_index(drop=True)

models_names = plot_df["Model"]

metrics = [
    ("Precision", "Precision_Mean", "Precision_Std"),
    ("Recall", "Recall_Mean", "Recall_Std"),
    ("F1 Score", "F1_Mean", "F1_Std"),
    ("ROC-AUC", "ROC_AUC_Mean", "ROC_AUC_Std"),
    ("PR-AUC", "PR_AUC_Mean", "PR_AUC_Std")
]

fig, axes = plt.subplots(
    1,
    5,
    figsize=(24, 6)
)

for ax, (title, mean_col, std_col) in zip(
    axes,
    metrics
):

    means = plot_df[mean_col]
    stds = plot_df[std_col]

    x = np.arange(len(models_names))

    ax.bar(
        x,
        means,
        yerr=stds,
        capsize=4
    )

    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(
        models_names,
        rotation=60,
        ha="right"
    )

    ax.set_ylim(0.85, 1.01)
    ax.set_ylabel("Score")
    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.suptitle(
    "7 Models — Chronological 5-Fold Cross-Validation",
    fontsize=16
)

plt.tight_layout()
plt.show()

### What this tells us

Since this is AML/fraud detection, I would not choose the model based only on F1 or Precision. Positive class is laundering, so PR-AUC is especially important because it measures how well the model ranks the rare positive class.

LightGBM is the strongest overall candidate.

Why?

- Highest PR-AUC: 0.95996
- Very high recall: 0.99939
- Very high F1: 0.94810
- ROC-AUC: 0.96066
- Its standard deviations are reasonably small, meaning performance is relatively consistent across the five chronological folds.

HistGradientBoosting has the highest ROC-AUC (0.96169), but LightGBM wins on the metric I'd prioritize for this highly imbalanced AML problem: PR-AUC.

### LightGBM threshold analysis

In [ ]:
# ============================================================
# LIGHTGBM — THRESHOLD ANALYSIS
# ============================================================

thresholds = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

# Get LightGBM model
lightgbm_model = trained_models["LightGBM"]

# Get probability predictions ONCE
y_validation_probability = (
    lightgbm_model.predict_proba(
        X_validation_processed
    )[:, 1]
)

threshold_results = []

for threshold in thresholds:

    # Convert probabilities to predictions
    y_prediction = (
        y_validation_probability >= threshold
    ).astype(int)

    # Metrics
    precision = precision_score(
        y_validation,
        y_prediction,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        y_prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        y_prediction,
        zero_division=0
    )

    pr_auc = average_precision_score(
        y_validation,
        y_validation_probability
    )

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "PR_AUC": pr_auc,
        "True_Negatives": tn,
        "False_Positives": fp,
        "False_Negatives": fn,
        "True_Positives": tp
    })

# Create DataFrame
lightgbm_threshold_df = pd.DataFrame(
    threshold_results
)

display(
    lightgbm_threshold_df.style
    .format({
        "Threshold": "{:.2f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "PR_AUC": "{:.4f}"
    })
)

This is actually a very interesting result. It tells us that LightGBM is extremely stable across a wide range of thresholds on this validation set.

In [ ]:
# Select LightGBM
lightgbm_model = trained_models["LightGBM"]

# Get probabilities
y_probability = lightgbm_model.predict_proba(
    X_validation_processed
)[:, 1]

# Apply threshold
threshold = 0.50

y_prediction = (
    y_probability >= threshold
).astype(int)

# Calculate confusion matrix
cm = confusion_matrix(
    y_validation,
    y_prediction,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print("=" * 60)
print("LIGHTGBM — CONFUSION MATRIX")
print("=" * 60)

print(f"\nThreshold: {threshold}")

print("\nConfusion Matrix:")
print(cm)

print("\nDetailed Results:")
print(f"True Negatives  (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives  (TP): {tp}")

# ============================================================
# VISUALIZE
# ============================================================

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Laundering"],
    yticklabels=["Normal", "Laundering"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(
    f"LightGBM Confusion Matrix (Threshold = {threshold})"
)

plt.show()

## 14. Load and prepare the test data

In [ ]:
# ============================================================
# LOAD TEST DATA
# ============================================================

print("=" * 60)
print("LOADING TEST DATA")
print("=" * 60)

test_parts = []

for i, file_path in enumerate(
    test_files,
    start=1
):
    print(
        f"Loading test file "
        f"{i}/{len(test_files)}"
    )

    df = pd.read_csv(file_path)

    test_parts.append(df)


# ------------------------------------------------------------
# Combine test files
# ------------------------------------------------------------

test_df = pd.concat(
    test_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# Convert timestamp and sort chronologically
# ------------------------------------------------------------

test_df["Timestamp"] = pd.to_datetime(
    test_df["Timestamp"]
)

test_df = test_df.sort_values(
    "Timestamp"
).reset_index(drop=True)


# ------------------------------------------------------------
# Separate features and target
# ------------------------------------------------------------

X_test = test_df[
    MODEL_NUMERIC_FEATURES
    + MODEL_CATEGORICAL_FEATURES
].copy()

y_test = test_df[
    TARGET_COLUMN
].copy()


# ------------------------------------------------------------
# Display test information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TEST DATA READY")
print("=" * 60)

print("Test shape:", test_df.shape)

print("X_test shape:", X_test.shape)

print("y_test shape:", y_test.shape)

print("\nTarget distribution:")
print(y_test.value_counts())

print("\nTarget percentages:")
print(
    y_test.value_counts(
        normalize=True
    ) * 100
)

print("\nDate range:")
print("Start:", test_df["Timestamp"].min())
print("End:", test_df["Timestamp"].max())

print(
    "\nMissing values:",
    X_test.isna().sum().sum()
)

In [ ]:
# ============================================================
# PREPROCESS TEST DATA
# ============================================================

print("=" * 60)
print("PREPROCESSING TEST DATA")
print("=" * 60)

# Transform using the already-fitted training preprocessor
X_test_processed = preprocessor.transform(
    X_test
)

# Convert to DataFrame with the SAME feature names as training
X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=PROCESSED_FEATURE_NAMES,
    index=X_test.index
)

print("\nTest preprocessing complete.")

print("Original test shape:",
      X_test.shape)

print("Processed test shape:",
      X_test_processed.shape)

print("Processed data type:",
      type(X_test_processed))

print("Number of feature names:",
      len(X_test_processed.columns))

print("\nFirst 10 feature names:")
print(
    X_test_processed.columns[:10].tolist()
)

In [ ]:
# ============================================================
# FINAL LIGHTGBM TEST EVALUATION
# ============================================================

print("=" * 70)
print("LIGHTGBM — FINAL TEST EVALUATION")
print("=" * 70)

# Get the trained LightGBM model
lightgbm_model = trained_models["LightGBM"]

# Evaluate on completely unseen test data
test_results = evaluate_model(
    model=lightgbm_model,
    X=X_test_processed,
    y=y_test,
    model_name="LightGBM",
    threshold=0.50
)

print("\n" + "=" * 70)
print("LIGHTGBM — FINAL TEST RESULTS")
print("=" * 70)

print(f"Threshold : {test_results['Threshold']:.2f}")
print(f"Precision : {test_results['Precision']:.4f}")
print(f"Recall    : {test_results['Recall']:.4f}")
print(f"F1 Score  : {test_results['F1']:.4f}")
print(f"ROC-AUC   : {test_results['ROC_AUC']:.4f}")
print(f"PR-AUC    : {test_results['PR_AUC']:.4f}")

print("\nConfusion Matrix Components:")
print(f"True Negatives  (TN): {test_results['True_Negatives']}")
print(f"False Positives (FP): {test_results['False_Positives']}")
print(f"False Negatives (FN): {test_results['False_Negatives']}")
print(f"True Positives  (TP): {test_results['True_Positives']}")

### The most important result

For an AML system, Recall = 99.87% is particularly important.

The model detected:

781 / 782 = 99.87% of laundering transactions

and missed only 1 out of 782 laundering transactions.

At the same time, the 91.67% precision means that when the model flags a transaction as laundering, the prediction is correct about 92% of the time.

In [ ]:
# Final LightGBM predictions
y_test_probability = lightgbm_model.predict_proba(
    X_test_processed
)[:, 1]

y_test_prediction = (
    y_test_probability >= 0.50
).astype(int)

# Confusion matrix
cm = confusion_matrix(
    y_test,
    y_test_prediction
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Laundering"],
    yticklabels=["Normal", "Laundering"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("LightGBM — Final Test Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINAL LIGHTGBM MODEL
# ============================================================

final_model = trained_models["LightGBM"]

print("Final model:")
print(final_model)

## 15. Save the model

In [ ]:
joblib.dump(
    final_model,
    "lightgbm_model.pkl"
)

print("LightGBM model saved successfully.")

### Save the preprocessor

In [ ]:
joblib.dump(
    preprocessor,
    "preprocessor.pkl"
)

print("Preprocessor saved successfully.")

### Save the processed feature names

In [ ]:
joblib.dump(
    PROCESSED_FEATURE_NAMES,
    "processed_feature_names.pkl"
)

print(
    "Processed feature names saved successfully."
)

print(
    "Number of features:",
    len(PROCESSED_FEATURE_NAMES)
)

### Save the model configuration

In [ ]:
MODEL_CONFIG = {
    "model_name": "LightGBM",
    "threshold": 0.50,
    "number_of_features": 80,
    "target_column": TARGET_COLUMN
}

joblib.dump(
    MODEL_CONFIG,
    "model_config.pkl"
)

print("Model configuration saved successfully.")
print(MODEL_CONFIG)

### Verify everything

In [ ]:
files_to_check = [
    "lightgbm_model.pkl",
    "preprocessor.pkl",
    "processed_feature_names.pkl",
    "model_config.pkl"
]

print("=" * 60)
print("SAVED MODEL FILES")
print("=" * 60)

for file_name in files_to_check:

    if os.path.exists(file_name):

        size = os.path.getsize(file_name) / (1024 * 1024)

        print(
            f"✓ {file_name} "
            f"({size:.2f} MB)"
        )

    else:

        print(
            f"✗ {file_name} NOT FOUND"
        )